## Analysis of DRB Salt and Bread data
### 11 November 2020

In [ ]:
from pathlib import Path

import numpy as np

import pandas as pd
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", None)

import matplotlib.pyplot as plt

import seaborn as sns
sns.set_style("whitegrid")

### HH Diary

In [ ]:
# Read in full 7-day diary dataset
fn = 'Diary_coicopV12.dta'
path = Path.cwd().joinpath('data', fn)
df = pd.read_stata(path)

In [ ]:
# Convert category to string
cat_cols = ['region', 'purch_cons', 'coicop', 'source', 'unit']

df[cat_cols] = df[cat_cols].astype(str)

# Sort values by region
df.sort_values(by='region', inplace=True)

In [ ]:
# Create smaller dataset of interest
salt = True

if salt:
    label = ['Salt']
else:
    label = ['Bread (white, brown, whole wheat, rye, maize, etc)']

print(f'The full dataset has {df.shape[0]} rows')
df = df[(df['label'].isin(label)) & (df['purch_cons'] == 'consumption')].reset_index(drop=True)
print(f'The {label} and consumption dataset has {df.shape[0]} rows')

In [ ]:
# Drop unnecessary columns
drop_cols = ['constituency', 'segment', 'du', 'hh', 'psu_id', 'coicop_others', 'psuno_f', 'stratum']

df = df.iloc[:, ~df.columns.isin(drop_cols)]

In [ ]:
print(f'In the {label} dataset, the freqeuncy of \'units\' is: \n\n {df.unit.value_counts()}')

In [ ]:
# Add grams value for units
if salt:
    grams_dict = {'teaspoon': 5.9, 'tablespoon': 17.07, 'grams': 1.0, 'pieces/number': np.nan, 'kilogram': 1000.0, 'millilitre': 1.0, 'litre': 1000.0}
else:
    grams_dict = {'teaspoon': 5.9, 'tablespoon': 17.07, 'grams': 1.0, 'pieces/number': np.nan, 'kilogram': 1000.0, 'millilitre': 1.0, 'litre': 1000.0, 'pieces/number': 35.0}

df['grams'] = [grams_dict[x] for x in df.unit]

# Update for single loaf
if salt:
    pass
else:
    df.loc[:, 'grams'] = np.where((df['unit'] == 'pieces/number') & (df['quantity'] <= 1), 700, df.grams)

In [ ]:
df.grams.value_counts(dropna=False)

In [ ]:
df.groupby('source').grams.describe()

In [ ]:
# Calculate grams consumed
df['grams_cons'] = df['quantity'] * df['grams']

# Calculate grams consumed per capita
df['grams_cons_cap'] = df['grams_cons'] / df['hhsize']

In [ ]:
df['grams_cons_cap'].describe()

## Clean extreme values

In [ ]:
# Remove extreme outliers
if salt:
    df = df.drop([16061])
else:
    df = df.drop([3794])

In [ ]:
df.insert(0, 'Total', 'Total')

In [ ]:
# List comp to create regional salt datasets
list_of_regions = df['region'].unique().tolist()

list_of_regional_dfs = [df[df['region'] == region] for region in list_of_regions]   

In [ ]:
# Function to create grams_cons_cap_df for each region
def grams_cons_cap(df, salt=True):
    # Drop missing values
    df.dropna(subset=['grams_cons_cap'], inplace=True)
    # Sort ascending
    df.sort_values(by='grams_cons_cap', inplace=True)

    # Generate cumulative sum
    df['grams_cons_cap_cumsum'] =  df['grams_cons_cap'].cumsum()

    # Percentiles
    df['grams_cons_cap_perc'] = pd.cut(df['grams_cons_cap_cumsum'], 100, labels=False)

    # Take median by percentile
    grams_cons_cap_perc_smooth = df.groupby('grams_cons_cap_perc').grams_cons_cap.mean().tolist()

    # Create dataset with percentiles from each
    if salt:
        label = 'Salt Consumption' + ' ' +'(g/c/d)'
    else:
        label = 'Bread Consumption' + ' ' +'(g/c/d)'
    
    grams_cons_cap_df = pd.DataFrame({
        label: grams_cons_cap_perc_smooth
    })

    grams_cons_cap_df.index = np.arange(1, len(grams_cons_cap_df)+1)
    # Truncate
    if salt:
        grams_cons_cap_df_trunc = grams_cons_cap_df[:80]
    else:
        grams_cons_cap_df_trunc = grams_cons_cap_df[25:75]

    return grams_cons_cap_df_trunc

In [ ]:
# List comp to create grams_cons_cap percentiles for each region
list_of_regional_grams_cons_cap = [grams_cons_cap(df) for df in list_of_regional_dfs]

# Total
total = grams_cons_cap(df)

In [ ]:
# Add total
list_of_regional_grams_cons_cap.insert(0, total)
list_of_regions.insert(0, 'Total')

# Dict with reigon name
region_grams_dict = dict(zip(list_of_regions, list_of_regional_grams_cons_cap))

In [ ]:
# Generate mean, median for each region
region_means = [df.mean().to_numpy() for df in list_of_regional_grams_cons_cap]
region_medians = [df.median().to_numpy() for df in list_of_regional_grams_cons_cap]

if salt:
    item_str = 'Salt'
    item_mean_str = 'Salt_mean'
    item_median_str = 'Salt_median'
else:
    item_str = 'Bread'
    item_mean_str = 'Bread_mean'
    item_median_str = 'Bread_median'

output = pd.DataFrame({
    'Region': list_of_regions,
    item_mean_str: region_means,
    item_median_str: region_medians
})

# Get numeric value
output[item_mean_str] = output[item_mean_str].str.get(0)
output[item_median_str] = output[item_median_str].str.get(0)

# Save output
fn = 'regional_' + item_str + '_est.csv'
path = Path.cwd().joinpath('output', fn)
output.to_csv(path, index=False)

In [ ]:
# Function to generate density charts
def generate_salt_chart(region, df):
    # Font parameters
    tw = {'fontname': 'Tw Cen MT'}
    cl = {'fontname': 'Calibri Light'}

    # Generate plot
    ax = plt.gca()

    df.plot.kde(ax=ax)

    plt.title(f'{region}: Distribution of Salt Consumption (g/c/d)', loc='left', fontsize=12, **tw)
    plt.xlabel("g/c/d", fontsize=8, **cl)
    plt.ylabel("Density", fontsize=8, **cl)
    plt.xticks(fontsize=8), plt.yticks(fontsize=9)
    plt.legend(fontsize=7, loc='upper left', frameon=False)
    ax.set_ylim([0,.25])
    fig = ax.get_figure()

    # Create filename
    location = 'output/charts/'
    f_ext = '.svg'
    chart_fn = region + '_' + 'salt_dist' + f_ext
    chart_path = os.path.join(os.getcwd(), location, chart_fn)
    # Save file
    fig.savefig(chart_path)
    plt.cla()

In [ ]:
[generate_salt_chart(region, df) for region, df in region_grams_dict.items()]